In [41]:
import recordlinkage
import pandas as pd
import re

social_dataset = pd.read_excel('../../Mediated Schema Excels/socialmedia_schema.xlsx')

social_dataset = social_dataset[
    social_dataset['Other'].notnull() & (social_dataset['Other'].str.strip() != '')
]

def tokenize_and_clean_url(url):
    if type(url) is not str:
        url = str(url)
    url = url.lower().replace("http://", "").replace("https://", "").replace("disfold.com/company","").replace("www","").rstrip("/")
    tokens = re.split(r'\W+', url)
    tokens = sorted(token for token in tokens if token)
    return " ".join(tokens)

social_dataset['Other_clean'] = social_dataset['Other'].apply(tokenize_and_clean_url)

def social_blocking():
    indexer = recordlinkage.Index()
    indexer.sortedneighbourhood('Name', window=5)
    candidate_links = indexer.index(social_dataset)
    compare = recordlinkage.Compare()
    compare.string('Other_clean', 'Other_clean', method='jarowinkler', label='url_similarity')
    compare_vectors = compare.compute(candidate_links, social_dataset)
    
    matched_pairs = compare_vectors[compare_vectors['url_similarity'] > 0.83]

    
    df = pd.DataFrame({
        "row_index_1": social_dataset.index.get_indexer(matched_pairs.index.get_level_values(0)) + 2,
        "row_index_2": social_dataset.index.get_indexer(matched_pairs.index.get_level_values(1)) + 2,
        "name_company_1": social_dataset.loc[matched_pairs.index.get_level_values(0), "Name"].values,
        "name_company_2": social_dataset.loc[matched_pairs.index.get_level_values(1), "Name"].values,
        "other_1": social_dataset.loc[matched_pairs.index.get_level_values(0), "Other"].values,
        "other_2": social_dataset.loc[matched_pairs.index.get_level_values(1), "Other"].values,
        "similarity_score": matched_pairs["url_similarity"].values,
        "is_match": 1
    })

    df.to_excel("../../blocking_excels/socialmedia_blocking.xlsx", index=False)

In [43]:
social_blocking()

In [46]:
url1 = "https://disfold.com/company/american-airlines-group-inc/"
url2 = "americanairlines.gcs-web.com"


url1_tok = tokenize_and_clean_url(url1)
url2_tok = tokenize_and_clean_url(url2)
print(url1_tok)
(url2_tok)

airlines american group inc


'americanairlines com gcs web'

In [47]:
import textdistance

threshold = 0.91

methods = {
    "jaro": textdistance.jaro,
    "jarowinkler": textdistance.jaro_winkler,
    "levenshtein": textdistance.levenshtein.normalized_similarity,
    "damerau_levenshtein": textdistance.damerau_levenshtein.normalized_similarity,
    "cosine": textdistance.cosine.normalized_similarity,
    "smith_waterman": textdistance.smith_waterman.normalized_similarity,
    "lcs": textdistance.lcsseq.normalized_similarity
}

for name, similarity_func in methods.items():
    sim = similarity_func(url1_tok, url2_tok)
    result = "Match" if sim >= threshold else "No match"
    print(f"{name}: similarity = {sim:.3f} -> {result}")

jaro: similarity = 0.715 -> No match
jarowinkler: similarity = 0.744 -> No match
levenshtein: similarity = 0.214 -> No match
damerau_levenshtein: similarity = 0.214 -> No match
cosine: similarity = 0.800 -> No match
smith_waterman: similarity = 0.222 -> No match
lcs: similarity = 0.464 -> No match
